# Aula 15 - PageRank, HITS e GNNs na Mão

Este notebook acompanha a Aula 15.

Objetivos:
- Implementar PageRank e HITS em grafos pequenos.
- Implementar camadas de GNN (GCN, GraphSAGE, GAT).
- Implementar uma GNN simples com MLP sobre vizinhos.
- Explorar o loop de message passing.

**Lacunas (TODO) para implementar**.

## 0. Setup

In [ ]:
!pip install networkx --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import random, math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Representando grafos

Vamos começar com um grafo dirigido simples de 4 nós.

In [ ]:
A = torch.tensor([
    [0., 1., 1., 0.],
    [0., 0., 1., 0.],
    [0., 0., 0., 1.],
    [1., 0., 0., 0.]
], dtype=torch.float32, device=device)

N = A.shape[0]
X_small = torch.eye(N, device=device)  # features identidade

A, X_small, N


### Exercício 1
1. Calcule o grau de saída de cada nó (soma por linha de `A`).
2. Crie a matriz de transição `P` normalizando as linhas de `A`.
3. Verifique que cada linha de `P` soma 1.

In [ ]:
# TODO: calcule o grau de saída (soma por linha)
# deg = ...

# TODO: crie a matriz de transição P normalizando por deg
# P = ...

# TODO: verifique se cada linha soma 1
# row_sums = ...
# print("Graus:", deg)
# print("Somas das linhas de P:", row_sums)


In [ ]:
def row_normalize(A):
    deg = A.sum(dim=-1, keepdim=True)
    deg = torch.where(deg == 0, torch.ones_like(deg), deg)
    return A / deg

P = row_normalize(A)
P_t = P.t()


## 2. PageRank como módulo linear

Vamos implementar `PageRankLayer` usando um `nn.Linear` com pesos fixos.

A atualização é:
`r_{k+1} = alpha * P^T r_k + (1-alpha) * 1/N`.

In [ ]:
class PageRankLayer(nn.Module):
    def __init__(self, P_t):
        super().__init__()
        N = P_t.shape[0]
        self.linear = nn.Linear(N, N, bias=False)
        # TODO 1: carregar P_t nos pesos da camada linear
        with torch.no_grad():
            # self.linear.weight.copy_( ... )
            ...

        # TODO 2: desativar gradientes (PageRank não é treinado)
        for p in self.linear.parameters():
            # p.requires_grad = ???
            ...

    def forward(self, r, alpha=0.85):
        # r: vetor de PageRank atual [N]
        # alpha: fator de amortecimento
        N = r.shape[0]
        # TODO 3: parte de caminhada aleatória
        # walk = self.linear( ... )
        ...

        # TODO 4: vetor de teleporte uniforme
        # teleport = ...
        ...

        # TODO 5: combine walk e teleport
        # return ...
        ...

pagerank_layer = PageRankLayer(P_t).to(device)

# TODO: inicialize r e rode algumas iterações
# r = ...
# for k in range(10):
#     r = pagerank_layer(r)
#     print(k, r)


## 3. HITS como módulos lineares

Agora vamos implementar o algoritmo HITS com duas camadas lineares fixas: uma com `A` e outra com `A^T`.

In [ ]:
class HITSLayer(nn.Module):
    def __init__(self, A):
        super().__init__()
        N = A.shape[0]
        self.A = nn.Linear(N, N, bias=False)
        self.A_t = nn.Linear(N, N, bias=False)
        with torch.no_grad():
            self.A.weight.copy_(A)
            self.A_t.weight.copy_(A.t())
        # TODO 1: desativar gradientes
        for p in self.parameters():
            # p.requires_grad = ???
            ...

    def forward(self, a, h):
        # a: vetor de authority [N]
        # h: vetor de hub [N]
        # TODO 2: atualizar a usando A^T e h
        # a_next = ...
        ...

        # TODO 3: normalizar a_next
        # a_next = ...
        ...

        # TODO 4: atualizar h usando A e a_next
        # h_next = ...
        ...

        # TODO 5: normalizar h_next
        # h_next = ...
        ...

        return a_next, h_next

hits_layer = HITSLayer(A).to(device)

# TODO: inicializar a, h e rodar algumas iterações
# a = ...
# h = ...
# for k in range(10):
#     a, h = hits_layer(a, h)
#     print(k, a, h)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------------------------------
# Funções de visualização para grafos e embeddings
# -------------------------------------------------------

def plot_graph_with_scores(G, scores=None, title="Grafo", cmap="viridis"):
    """
    G: grafo networkx
    scores: tensor/array de shape [N] com um valor escalar por nó (ex: PageRank, authority, etc.)
    """
    plt.figure(figsize=(6, 6))
    pos = nx.spring_layout(G, seed=0)

    if scores is not None:
        scores_np = scores.detach().cpu().numpy() if hasattr(scores, "detach") else np.array(scores)
        vmin, vmax = scores_np.min(), scores_np.max()
        nx.draw(
            G, pos,
            node_color=scores_np,
            cmap=cmap,
            with_labels=True,
            node_size=600,
            vmin=vmin,
            vmax=vmax
        )
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
        sm.set_array([])
        plt.colorbar(sm, label="Score")
    else:
        nx.draw(
            G, pos,
            with_labels=True,
            node_color="lightblue",
            node_size=600
        )

    plt.title(title)
    plt.axis("off")
    plt.show()


def plot_embeddings_2d(H, labels=None, title="Embeddings 2D (PCA)"):
    """
    H: tensor [N, d] com embeddings de nós
    labels: tensor/array [N] com rótulos inteiros (ex: comunidade)
    Faz PCA para projetar em 2D e plota scatter.
    """
    H_np = H.detach().cpu().numpy() if hasattr(H, "detach") else np.array(H)

    # PCA manual simples para 2D
    H_centered = H_np - H_np.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(H_centered, full_matrices=False)
    H_2d = H_centered @ Vt[:2].T

    plt.figure(figsize=(6, 6))
    if labels is not None:
        labels_np = labels.detach().cpu().numpy() if hasattr(labels, "detach") else np.array(labels)
        scatter = plt.scatter(H_2d[:, 0], H_2d[:, 1], c=labels_np, cmap="tab10", s=80)
        plt.legend(*scatter.legend_elements(), title="Label")
    else:
        plt.scatter(H_2d[:, 0], H_2d[:, 1], s=80)

    for i, (x, y) in enumerate(H_2d):
        plt.text(x + 0.02, y + 0.02, str(i), fontsize=9)

    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(alpha=0.2)
    plt.show()



# 1) Visualizar PageRank no grafo pequeno de 4 nós
# plot_graph_with_scores(
#     G=nx.from_numpy_array(A.detach().cpu().numpy(), create_using=nx.DiGraph),
#     scores=r,
#     title="PageRank no grafo toy"
# )

# 2) Visualizar HITS (authority) no grafo toy
# plot_graph_with_scores(
#     G=nx.from_numpy_array(A.detach().cpu().numpy(), create_using=nx.DiGraph),
#     scores=a,
#     title="Authority (HITS)"
# )


## 4. Camadas de GNN: GCN, GraphSAGE, GAT

Vamos criar camadas de GNN. A GCN e a GAT estão completas, a GraphSAGE terá TODOs (na versão de alunos).

In [ ]:
def gcn_norm(A):
    I = torch.eye(A.shape[0], device=A.device)
    A_tilde = A + I
    deg = A_tilde.sum(dim=-1)
    D_inv_sqrt = torch.diag(torch.pow(deg, -0.5))
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, X, A_hat):
        H = A_hat @ X
        return F.relu(self.linear(H))


In [ ]:
class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(2 * in_dim, out_dim)

    def forward(self, X, A):
        # X: [N, F]
        # A: [N, N]
        # TODO 1: graus = soma por linha
        # deg = ...
        ...

        # TODO 2: evitar divisão por zero
        # deg = deg.clamp(min=1.)
        ...

        # TODO 3: média dos vizinhos: neigh_mean = (A @ X) / deg
        # neigh_mean = ...
        ...

        # TODO 4: concatenar X e neigh_mean
        # H_cat = torch.cat([...], dim=-1)
        ...

        # TODO 5: aplicar linear + ReLU
        # return F.relu(self.linear(H_cat))
        ...


In [ ]:
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Linear(2 * out_dim, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, X, A):
        H = self.W(X)
        N = H.size(0)
        H_i = H.unsqueeze(1).repeat(1, N, 1)
        H_j = H.unsqueeze(0).repeat(N, 1, 1)
        e_ij = self.leaky_relu(self.a(torch.cat([H_i, H_j], dim=-1))).squeeze(-1)
        e_ij = e_ij.masked_fill(A == 0, float('-inf'))
        alpha = torch.softmax(e_ij, dim=-1)
        H_prime = alpha @ H
        return F.relu(H_prime)


## 5. SimpleGNNLayer e loop de message passing

Implementar uma GNN simples que faz agregação (média dos vizinhos) e update via um MLP.

In [ ]:
class SimpleGNNLayer(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, X, A):
        # X: [N, F]
        # A: [N, N]
        # TODO 1: graus = soma por linha
        # deg = ...
        ...

        # TODO 2: evitar divisão por zero
        # deg = deg.clamp(min=1.)


        # TODO 3: média dos vizinhos m = (A @ X) / deg
        # m = ...


        # TODO 4: concatenar X e m
        # H_cat = torch.cat([...], dim=-1)


        # TODO 5: aplicar MLP
        # return self.mlp(H_cat)


# TODO: teste o loop de message passing com SimpleGNNLayer
# layer = SimpleGNNLayer(in_dim=..., hidden_dim=16, out_dim=...).to(device)
# H = X_small.clone()
# for k in range(5):
#     H = layer(H, A)
#     print(k, H)


## 6. Experimento: Karate Club (classificação de nós)

Monte um modelo de 2 camadas (GCN, GraphSAGE ou GAT) para classificar os nós do grafo Karate Club em duas comunidades.

In [ ]:
G_karate = nx.karate_club_graph()
N_k = G_karate.number_of_nodes()
A_k_np = nx.to_numpy_array(G_karate)
A_k = torch.tensor(A_k_np, dtype=torch.float32, device=device)
X_k = torch.eye(N_k, device=device)

labels = []
for i in range(N_k):
    club = G_karate.nodes[i]['club']
    labels.append(0 if club == 'Mr. Hi' else 1)
y_k = torch.tensor(labels, dtype=torch.long, device=device)

idx_all = list(range(N_k))
random.seed(0)
random.shuffle(idx_all)
split = N_k // 2
idx_train = torch.tensor(idx_all[:split], dtype=torch.long, device=device)
idx_test = torch.tensor(idx_all[split:], dtype=torch.long, device=device)

A_k_hat = gcn_norm(A_k)

def train_node_clf(model, X, y, idx_train, idx_test, epochs=200, lr=0.01):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        logits = model(X)
        loss = F.cross_entropy(logits[idx_train], y[idx_train])
        loss.backward()
        opt.step()
        if (epoch + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                logits = model(X)
                pred = logits.argmax(dim=-1)
                acc_train = (pred[idx_train] == y[idx_train]).float().mean().item()
                acc_test = (pred[idx_test] == y[idx_test]).float().mean().item()
            print(f"Epoch {epoch+1:03d} | loss={loss.item():.4f} | "
                  f"train acc={acc_train:.3f} | test acc={acc_test:.3f}")
    return model

# Exemplo com GCN de 2 camadas
class GCNNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, A_hat):
        super().__init__()
        self.A_hat = A_hat
        self.g1 = GCNLayer(in_dim, hidden_dim)
        self.g2 = GCNLayer(hidden_dim, out_dim)

    def forward(self, X):
        H = self.g1(X, self.A_hat)
        H = F.dropout(H, p=0.5, training=self.training)
        H = self.g2(H, self.A_hat)
        return H

gcn_model = GCNNet(in_dim=N_k, hidden_dim=16, out_dim=2, A_hat=A_k_hat)
gcn_model = train_node_clf(gcn_model, X_k, y_k, idx_train, idx_test)


In [ ]:

# 3) Visualizar Karate Club com comunidades
# plot_graph_with_scores(
#     G_karate,
#     scores=y_k,
#     title="Karate Club - Comunidades (rótulos)"
# )

# 4) Visualizar embeddings aprendidos (por exemplo, saída da penúltima camada de uma GNN)
# with torch.no_grad():
#     H_embed = gcn_model.g1(X_k, A_k_hat)  # embeddings intermediários
# plot_embeddings_2d(H_embed, labels=y_k, title="Embeddings GCN - Karate Club")
